# AFS Bootstrap

**Run once** to create the `AUDITED_FINANCIALS` database, all `COMMON` schema objects, and the `@AFS_STAGE` internal stage.

Re-running is safe — all DDL uses `CREATE IF NOT EXISTS`.

In [ ]:
# Snowflake Notebooks provide `session` automatically.
# Verify we are in the right context.
print('Account  :', session.get_current_account())
print('User     :', session.get_current_user())
print('Warehouse:', session.get_current_warehouse())

In [ ]:
bootstrap_sql = open('../sql/bootstrap.sql').read()

# Execute each statement individually (Snowflake SQL API is single-statement)
for stmt in bootstrap_sql.split(';'):
    s = stmt.strip()
    if s:
        session.sql(s).collect()
        print(f'OK: {s[:80].replace(chr(10), " ")}...')

print('\nBootstrap complete.')

In [ ]:
# Seed STANDARD_TAXONOMY from CSV
import pandas as pd

taxonomy = pd.read_csv('../sql/standard_taxonomy_seed.csv')
session.write_pandas(
    taxonomy,
    table_name='STANDARD_TAXONOMY',
    database='AUDITED_FINANCIALS',
    schema='COMMON',
    overwrite=False,
    auto_create_table=False,
)
print(f'Seeded {len(taxonomy)} taxonomy rows.')

In [ ]:
# Verify key objects exist
checks = [
    "SELECT COUNT(*) FROM AUDITED_FINANCIALS.COMMON.STANDARD_TAXONOMY",
    "SHOW STAGES LIKE 'AFS_STAGE' IN SCHEMA AUDITED_FINANCIALS.COMMON",
    "SELECT COUNT(*) FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING",
]
for q in checks:
    result = session.sql(q).collect()
    print(q[:60], '->', result[0][0] if result else 'OK')